In [1]:
import noisereduce as nr
import numpy as np
from scipy.io import wavfile

# 1. Cargar el audio
samplerate, data = wavfile.read('Prueba.wav')

# Si es estéreo, tomamos solo el primer canal
if len(data.shape) > 1:
    data = data[:, 0]

# 2. Reducción de ruido por "Spectral Gating" (Método moderno)
# El parámetro stationary=False hace que el algoritmo use un modelo dinámico
# que se adapta mejor a voces humanas y ruidos cambiantes (más parecido a Discord)
audio_limpio = nr.reduce_noise(y=data, sr=samplerate, stationary=False)

# 3. Guardar el resultado
wavfile.write('audio_discord_style.wav', samplerate, audio_limpio)

print("¡Procesamiento avanzado completado! Escucha 'audio_discord_style.wav'")

/Users/diegobenjaminhurtadoramos/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


¡Procesamiento avanzado completado! Escucha 'audio_discord_style.wav'


In [2]:
# Calcular FFT
fhat = np.fft.fft(audio_norm, n)
PSD = fhat * np.conjugate(fhat) / n

freq = (1 / (dt * n)) * np.arange(n)
L = np.arange(1, int(np.floor(n / 2)), dtype='int')

# Graficar para buscar el umbral
plt.figure(figsize=(10, 4))
plt.plot(freq[L], PSD[L].real, color='red')
plt.title('Espectro de Frecuencias (Busca los picos de ruido)')
plt.xlabel('Frecuencia (Hz)')
plt.ylabel('Potencia')
plt.xlim(0, 8000) # Ajusta este valor si el ruido es más agudo
plt.grid(True)
plt.show()

NameError: name 'audio_norm' is not defined

In [19]:
# ==========================================
# NUEVA CELDA 3: FILTRO PASA-BANDA PARA VOCES
# ==========================================

# Definimos el rango de frecuencias de la voz humana (en Hertz)
FREQ_MIN = 150   # Corta graves profundos y ruidos de motor
FREQ_MAX = 1000  # Corta brillos agudos, platillos y siseos

# Obtener las frecuencias reales (positivas y negativas) generadas por la FFT
freqs_completas = np.fft.fftfreq(n, d=dt)

# Creamos una copia del espectro original
fhat_clean = fhat.copy()

# Filtro: Ponemos en 0 todo lo que esté FUERA del rango de la voz
# Usamos np.abs para aplicar el corte a las frecuencias negativas simétricas de Fourier
fhat_clean[np.abs(freqs_completas) < FREQ_MIN] = 0
fhat_clean[np.abs(freqs_completas) > FREQ_MAX] = 0

# Reconstruir audio (IFFT)
audio_filtered = np.fft.ifft(fhat_clean)
audio_filtered_real = audio_filtered.real

# Volver a escalar al formato de audio .wav (16-bit)
audio_filtered_scaled = np.int16(audio_filtered_real * 32767 / np.max(np.abs(audio_filtered_real)))

# Guardar nuevo archivo
wavfile.write('audio_solo_voces.wav', samplerate, audio_filtered_scaled)
print(f"¡Filtro pasa-banda aplicado! Se aislaron frecuencias entre {FREQ_MIN}Hz y {FREQ_MAX}Hz.")

¡Filtro pasa-banda aplicado! Se aislaron frecuencias entre 150Hz y 1000Hz.


In [3]:
fig, axs = plt.subplots(2, 1, figsize=(10, 6))

axs[0].plot(t, audio_norm, color='red', alpha=0.6, label='Con Ruido')
axs[0].plot(t, audio_filtered_real, color='blue', alpha=0.8, label='Filtrado')
axs[0].set_title('Forma de Onda en el Tiempo')
axs[0].legend()

PSD_clean = fhat_clean * np.conjugate(fhat_clean) / n
axs[1].plot(freq[L], PSD_clean[L].real, color='blue')
axs[1].set_title('Espectro Filtrado')
axs[1].set_xlim(0, 8000)

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined